# F02-P5 Benefit

**Benefit Quantification, components 5.x.**

Turns the site description produced by the earlier stages into the benefit a project could
claim. Where F02-P2 says what is there and F02-P4 says what may be done with it, this notebook
says what that is worth in carbon terms over a project lifetime.

> **Not runnable yet.** All analysis logic below is real Python. Only file access is stubbed.

**Status.** 5.1 Avoided Emissions from Unplanned Deforestation is written. Later components are not
started. The three NBS pathways carry different benefit logic and only the first is specified:

| Pathway | Benefit logic | Status |
|---|---|---|
| Protect | avoided loss of standing carbon | 5.1, written |
| Manage | reduced degradation, slower loss | not started |
| Restore | removals, growth towards a reference stock | not started |

## Project duration enters here

This is the first component in the tool that depends on a user input other than the AOI
polygon. `PROJECT_DURATION_YEARS` is set in the Setup cell below, next to `AOI_PATH`, following
the same pattern. It is passed as a function argument rather than read as a global, and it is
written into `values` so that a saved result records the duration it was produced with.

## Handoff

Reads `outputs/<aoi_id>__F02-P2-general.json` for the historical deforestation rate from 1.5.
This stage is **required**, not optional: without a rate there is no baseline and 5.1 has
nothing to project. Writes `outputs/<aoi_id>__F02-P5-benefit.json`.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import math
from dataclasses import dataclass

import geopandas as gpd
import numpy as np

from config import *
from common import *

In [ ]:
AOI_PATH = r"<SET: path to the project AOI polygon>"
aoi_id = "<SET: short id for this run, must match the F02-P2 and F02-P4 notebooks>"

# User input. Whole years, the crediting period the user is asking about.
PROJECT_DURATION_YEARS = 20   # <SET: project duration in years>

aoi = prepare_aoi(gpd.read_file(AOI_PATH))
print(f"AOI {aoi_id}: {fmt_ha(aoi.area_ha)} over {PROJECT_DURATION_YEARS} years")

---
## 5.1 Avoided Emissions from Unplanned Deforestation

**Why "unplanned".** This is the AUD case: deforestation driven by diffuse, unsanctioned pressure,
which is why the baseline comes from a spatial risk model rather than from a document. Avoided
planned deforestation (APD) is a different construct with a different baseline, a land status
overlay showing legally sanctioned conversion, and it is not what this component estimates. The
title says unplanned so the two are not read as interchangeable.

The component is computed on the Protect area, but Protect is the input it runs on, not what it
measures. The output is an avoided emission estimate.

Estimates how much CO2e a Protect project could keep out of the atmosphere over its lifetime, by
projecting the historical deforestation rate forward, placing that projected loss on the highest
risk forest, and reading the carbon standing on it.

**Data.** No new layer. Three layers that other components already declare, read onto one grid:

| Layer | Role | Declared by |
|---|---|---|
| `pathway.tif` band 1 | which pixels are Protect (code 1) | 4.1 |
| `pathway.tif` band 3 | reference ecosystem, for the peat check | 4.1 |
| `prob.tif` | ranks forest pixels by deforestation risk | 1.6 |
| `agb_mgha.tif`, `bgb_mgha.tif` | carbon standing on each pixel | 3.1 |

Plus one number from an earlier stage: `rate_pct` from 1.5.

### The four steps

```
1. protect_ha    = area of pixels that are Protect AND carry a risk value
2. lost_ha(t)    = protect_ha * (1 - exp(-r * t))        r = rate_pct / 100
3. allocate      = take pixels in descending risk order until lost_ha(t) is filled
4. avoided(t)    = sum of (AGB + BGB) on those pixels * 0.47 * 44/12
```

### Decisions locked

**The rate is borrowed, and it has to be.** Protect is assigned upstream to forest that
persisted from 2014 to 2024. Measuring historical loss inside the Protect area therefore returns
zero by construction, not by observation: the area was selected on the criterion of not having
been deforested. This is a selection effect, so its own history carries no information about its
future and the rate must come from a wider population of forest.

**That wider population is the AOI forest, from 1.5.** The team chose the rate the tool already
has over a precomputed district reference table. Two consequences to keep in view:

1. The AOI average includes degraded and edge forest, which loses faster than intact interior.
   Applying it to Protect leans towards over-estimating the baseline.
2. The rate is measured inside a polygon the user draws. Adding already cleared land to the AOI
   raises `r`, which is then charged to the Protect area. Nothing in the calculation resists
   this.

`values["baseline_rate_source"]` records which population the rate came from, so a later switch
to a district table is visible in saved results rather than silent. `_project_loss_series` takes
the rate as an argument for the same reason.

**Projection is compounding, not linear.** `lost_ha(t) = protect_ha * (1 - exp(-r * t))`. The
Puyravaud rate in 1.5 is an exponential rate, so multiplying a constant ha/year by the duration
would impose a linear shape on an exponential process and over-project long durations. The
compounding form also approaches the Protect area asymptotically instead of crossing it, so the
cap on projected loss is a safeguard rather than something that fires routinely.

**Allocation is by descending risk, and the result of that is conservative.** The highest risk
pixels sit on frontiers and edges, and frontier forest usually carries less biomass than intact
interior. Ranked allocation therefore yields a **lower** figure than spreading the same loss
evenly across the Protect area. That is a property of the method, not a bug, and the difference
is reported in `values["uniform_allocation_tco2e"]` so the two can be compared.

Two limits on the ranking, from the 1.6 markdown cell: `prob.tif` is a relative spatial ranking
and not an absolute probability, and if it is a mosaic of separately fitted regional models the
scale may not be comparable across regions. Both are acceptable here because this component uses
only the **order** of pixels within one AOI, never the level, and never compares one AOI to
another. Risk ties are broken arbitrarily by the sort, which is harmless: tied pixels are
interchangeable by definition of the ranking.

**The annual series can rise, and that is not a bug.** Two quantities move in opposite
directions as the projection runs. The area lost each year falls, because compounding works on a
shrinking stock. The carbon per hectare of that loss rises, because the allocation walks down the
risk ranking and lower risk forest is usually less degraded. Their product has no guaranteed
direction. On a synthetic site with a degraded frontier the annual figure declines gently inside
each stretch of similar forest, then steps up whenever the ranking crosses into denser forest.
Only two things are guaranteed: the annual area lost declines monotonically, and the cumulative
total rises. Nobody should later smooth or "correct" the annual curve into a decline.

**The pixel at the boundary is split, not rounded.** The projected area rarely lands on a whole
pixel. The last pixel contributes the fraction of its area that is needed. Rounding to whole
pixels instead would make the annual series step rather than curve on small sites.

**Emission factor is 100 percent of AGB plus BGB, labelled as an upper bound.** No assumption is
made about what replaces the forest. IPCC and VCS practice takes the difference between the
forest stock and the stock of the land use that follows, which usually leaves 5 to 15 percent
standing. Taking the full stock avoids a parameter that is not calibrated, at the cost of
sitting at the top of the plausible range. The narrative says so.

**No deductions.** Leakage, uncertainty and the non-permanence buffer are all absent. A VCS
buffer alone is commonly 10 to 25 percent. The figure is gross and is **not a creditable
volume**. The team decided this belongs in the documentation only, so the component carries no
`deductions_applied` marker in `values` and the narrative makes no claim about it either. Anyone
reading `total_tco2e` out of the result JSON has to know from here that it is gross. Read the
next section before quoting the number anywhere.

**The narrative names the ecosystem, and it lists every one present.** The word comes from
pathway band 3 restricted to the Protect pool, ordered by area, joined with `oxford_join`, so a
mixed site reads "this forest and peatland ecosystem" rather than being given a single label.
This follows the rule the rest of the tool uses: the AOI is heterogeneous and is never reduced
to one class. Only three words can appear, because `prob.tif` is forest masked and Protect pixels
on grassland, savanna or water carry no risk value and never reach the pool. If one does reach
it, the risk layer and band 3 disagree about what forest is, and the component flags it rather
than inventing a fourth word.

**Grid alignment is required here, unlike in 3.1.** Component 3.1 integrates AGB and BGB
separately on purpose, so the two rasters never have to share a grid. 5.1 cannot do that: it
selects pixels by one raster and reads carbon at the pixels it selected, so risk, carbon and
pathway must describe the same ground cell by cell. Every read passes `like=risk`, which puts
all of them on the `prob.tif` grid. Consequence worth expecting: the Protect area measured here
is at risk-layer resolution and can differ by a fraction of a percent from the Protect area
reported by 4.1, which is measured on the pathway grid. The two are not errors of each other.

**Nodata biomass counts as zero**, the same rule 3.1 uses, with coverage measured and flagged.

### What this number leaves out

- **Peat.** On a peatland reference ecosystem the avoided emission is dominated by peat
  oxidation and fire, which can exceed the biomass pools several times over. 5.1 sees only AGB
  and BGB, so a peat Protect area is under-estimated by a wide margin. This is flagged hard, not
  footnoted, using pathway band 3.
- **Deadwood and litter**, inherited from 3.1.
- **Degradation without clearing.** The baseline is deforestation only. Forest that stays forest
  while losing carbon is a Manage question and belongs in a later component.

### Open items

1. The rate is extrapolated beyond the ten year window it was measured in whenever the duration
   exceeds `BASELINE_RATE_MAX_YEARS`. The tool flags this and still returns the figure. A real
   baseline would be reassessed periodically instead.
2. Timing within a year is ignored. Loss is treated as occurring at year end, and carbon as
   released in full at the moment of clearing. Belowground carbon actually decays over several
   years, which matters for a year by year credit schedule but not for a lifetime total.
3. The uniform allocation comparison is reported but not used. If the gap between ranked and
   uniform allocation turns out to be large on real sites, it is a measure of how much the
   estimate depends on the risk layer being right, and may deserve a place in the narrative.

### Example render

Single ecosystem:

> Protecting this forest ecosystem can avoid an estimated 612,000 tonnes of CO2eq emissions over
> the project's 20 year duration.

Mixed Protect area:

> Protecting this forest and peatland ecosystem can avoid an estimated 612,000 tonnes of CO2eq
> emissions over the project's 20 year duration.

No historical loss, so no baseline:

> No forest loss was recorded in this project area between 2014 and 2024, so the baseline
> projects no further loss and no avoided emissions can be claimed from protecting the standing
> forest.

| Year | Cumulative loss (ha) | Avoided this year (tCO2e) | Cumulative (tCO2e) |
|---|---|---|---|
| 1 | 14 | 26,400 | 26,400 |
| 2 | 27 | 26,100 | 52,500 |
| ... | | | |
| 19 | 246 | 34,500 | 577,800 |
| 20 | 258 | 34,200 | 612,000 |

Note the annual column rising towards the end. That is the ranking reaching denser forest, and
it is explained above.

**Narrative specified by the team**, 2026-07-21. It is deliberately one sentence and carries the
headline figure and the duration only. Everything the previous placeholder said about area,
projected loss, pools and deductions has been removed from the narrative. Most of it is still in
`values`; the deductions marker is not, by the same decision. The frontend can build any
supporting line it needs from `values`, but nothing in the component says the number is gross.

### Downstream use

The lifetime total is the headline of the Protect side of Phase 5, and feeds the Climate
Resilience and Mitigation pillar of the Triple Win framework. The annual series is chart data for
the frontend. The Manage and Restore components will produce figures on the same units and the
same duration, so the three can be presented side by side without conversion.

In [ ]:
# Repeated rather than imported: notebooks cannot import each other, and the Climate notebook
# owns the same tuple for 3.1. Keep the two in step if a pool is ever added.
BIOMASS_POOLS = ("Aboveground biomass", "Belowground biomass")


@dataclass(frozen=True)
class ProjectionYear:
    """One year of the baseline projection."""

    year: int
    cumulative_loss_ha: float
    annual_avoided_tco2e: float
    cumulative_avoided_tco2e: float


def _project_loss_series(area_ha: float, rate_frac: float, years: int) -> list[float]:
    """Cumulative area lost by the end of each year, compounding.

    Takes the rate as an argument rather than reading it from a fixed source, so that swapping
    the AOI rate for a district reference table later changes the caller, not this function.
    Compounding form: the remaining stock shrinks each year, so annual loss declines. The series
    approaches `area_ha` and never exceeds it.
    """
    return [area_ha * (1.0 - math.exp(-rate_frac * t)) for t in range(1, years + 1)]


def _cumulative_carbon_by_rank(
    density_tco2e_ha: np.ndarray, risk: np.ndarray, pixel_area_ha: float
) -> np.ndarray:
    """Carbon accumulated as pixels are taken in descending risk order.

    Returns an array where element k is the total tCO2e on the k+1 highest risk pixels. Sorting
    once here is what makes the annual series cheap: every year is a lookup into this curve
    rather than a new pass over the raster.
    """
    order = np.argsort(risk)[::-1]
    return np.cumsum(density_tco2e_ha[order]) * pixel_area_ha


def _carbon_on_area(cumulative: np.ndarray, area_ha: float, pixel_area_ha: float) -> float:
    """Carbon on the highest risk `area_ha`, splitting the pixel that straddles the boundary.

    Whole pixels first, then the fraction of the next pixel needed to reach the target area.
    Splitting rather than rounding keeps the annual series smooth on small sites, where one
    pixel can be a visible share of a year of projected loss.
    """
    if area_ha <= 0 or cumulative.size == 0:
        return 0.0

    total_ha = cumulative.size * pixel_area_ha
    if area_ha >= total_ha:
        return float(cumulative[-1])

    whole = int(area_ha // pixel_area_ha)
    carbon = float(cumulative[whole - 1]) if whole > 0 else 0.0

    remainder_ha = area_ha - whole * pixel_area_ha
    if remainder_ha > 0 and whole < cumulative.size:
        prev = float(cumulative[whole - 1]) if whole > 0 else 0.0
        next_pixel_carbon = float(cumulative[whole]) - prev
        carbon += next_pixel_carbon * (remainder_ha / pixel_area_ha)

    return carbon


def analyze_avoided_deforestation_emissions(
    aoi: AOI, duration_years: int, rate_pct: float | None
) -> ComponentResult:
    """Component 5.1. Avoided emissions from unplanned deforestation on the Protect area, in tCO2e.

    `rate_pct` is the annual deforestation rate from 1.5, in percent, measured over the whole
    AOI forest. See the markdown cell for why the Protect area cannot supply its own rate.
    """
    component = "5.1 Avoided Emissions from Unplanned Deforestation"

    if duration_years < 1:
        raise ValueError("PROJECT_DURATION_YEARS must be a whole number of years, at least 1.")

    # The risk layer defines the working grid. It is already forest masked upstream, and it is
    # the layer the allocation ranks on, so everything else is aligned to it.
    risk_slice = load_raster_clipped(PROB_RASTER, aoi, resampling="nearest")
    pathway = load_raster_clipped(
        PATHWAY_RASTER, aoi, resampling="nearest", band=PATHWAY_BAND, like=risk_slice
    )

    pixel_area_ha = risk_slice.pixel_area_ha

    # Protect pixels that also carry a risk value. Protect on a non forest reference ecosystem,
    # grassland or savanna for instance, has no risk value because prob.tif is forest masked,
    # and cannot receive projected deforestation.
    protect_all = (pathway.values == PROTECT_CODE).filled(False)
    pool = protect_all & ~np.ma.getmaskarray(risk_slice.values)

    protect_all_ha = int(protect_all.sum()) * pixel_area_ha
    protect_ha = int(pool.sum()) * pixel_area_ha

    if protect_ha <= 0:
        return not_applicable(
            component,
            "No forest in this project area falls under the Protect pathway, so avoided "
            "emissions from deforestation cannot be estimated.",
        )

    if rate_pct is None:
        return not_applicable(
            component,
            "No historical deforestation rate is available for this project area, so a "
            "baseline for avoided emissions cannot be projected.",
        )

    flags: list[str] = []

    risk_coverage_pct = safe_pct(protect_ha, protect_all_ha)
    if risk_coverage_pct < PROTECT_RISK_COVERAGE_WARN_PCT:
        flags.append(
            f"5.1: the risk layer covers only {risk_coverage_pct:.0f}% of the Protect area. "
            "The remainder carries no projected loss and no avoided emissions."
        )

    if duration_years > BASELINE_RATE_MAX_YEARS:
        flags.append(
            f"5.1: the deforestation rate was measured over {DEFOR_PERIOD_YEARS} years and is "
            f"projected over {duration_years}. A rate that far outside its measurement window "
            "is an assumption, not an observation."
        )

    # Pathway band 3 does two jobs here, from one pass: it supplies the ecosystem word the
    # narrative needs, and it locates peatland.
    ecosystem = load_raster_clipped(
        PATHWAY_RASTER, aoi, resampling="nearest", band=PATHWAY_ECOSYSTEM_BAND, like=risk_slice
    )
    codes, counts = np.unique(ecosystem.values.filled(0)[pool].astype(int), return_counts=True)
    ecosystem_ha = {int(c): int(n) * pixel_area_ha for c, n in zip(codes, counts)}

    # Named in descending area, so the reading order matches what the site is mostly made of.
    ecosystem_words = [
        PROTECT_ECOSYSTEM_WORDS[c]
        for c in sorted(ecosystem_ha, key=ecosystem_ha.get, reverse=True)
        if c in PROTECT_ECOSYSTEM_WORDS
    ]
    ecosystem_label = oxford_join(ecosystem_words) or "natural"

    # A pool pixel outside the mapping means prob.tif and band 3 disagree about what is forest.
    unmapped_ha = sum(ha for c, ha in ecosystem_ha.items() if c not in PROTECT_ECOSYSTEM_WORDS)
    if unmapped_ha > 0:
        flags.append(
            f"5.1: {fmt_ha(unmapped_ha)} of the Protect area carries a risk value but a "
            "reference ecosystem that is not forest, mangrove or peatland. The risk layer and "
            "pathway band 3 disagree about what is forest."
        )

    peat_ha = ecosystem_ha.get(PATHWAY_ECOSYSTEM_PEATLAND, 0.0)
    if peat_ha > 0:
        flags.append(
            f"5.1: {fmt_ha(peat_ha)} of the Protect area sits on peatland. Only aboveground and "
            "belowground biomass is counted, and on peat the avoided emission is dominated by "
            "peat oxidation, so this figure is a large under-estimate there."
        )

    # Carbon density per pixel, tCO2e per hectare. Nodata is zero biomass, as in 3.1.
    agb = load_raster_clipped(AGB_RASTER, aoi, resampling="average", like=risk_slice)
    bgb = load_raster_clipped(BGB_RASTER, aoi, resampling="average", like=risk_slice)

    biomass_mgha = agb.values.filled(0.0).astype(float) + bgb.values.filled(0.0).astype(float)
    density = biomass_mgha * CARBON_FRACTION * CO2_PER_C

    biomass_coverage_pct = safe_pct(
        int((pool & ~np.ma.getmaskarray(agb.values)).sum()) * pixel_area_ha, protect_ha
    )
    if biomass_coverage_pct < CARBON_COVERAGE_WARN_PCT:
        flags.append(
            f"5.1: the biomass rasters cover only {biomass_coverage_pct:.0f}% of the Protect "
            "area. Nodata counts as zero carbon, so the estimate is an under-estimate by an "
            "unknown amount."
        )

    pool_density = density[pool]
    pool_risk = risk_slice.values.filled(0)[pool].astype(float)
    cumulative = _cumulative_carbon_by_rank(pool_density, pool_risk, pixel_area_ha)

    standing_tco2e = float(cumulative[-1])

    rate_frac = rate_pct / 100.0
    loss_series = _project_loss_series(protect_ha, rate_frac, duration_years)

    rows: list[ProjectionYear] = []
    previous_carbon = 0.0
    for year, cumulative_loss_ha in enumerate(loss_series, start=1):
        capped_ha = min(cumulative_loss_ha, protect_ha)
        carbon = _carbon_on_area(cumulative, capped_ha, pixel_area_ha)
        rows.append(
            ProjectionYear(
                year=year,
                cumulative_loss_ha=capped_ha,
                annual_avoided_tco2e=carbon - previous_carbon,
                cumulative_avoided_tco2e=carbon,
            )
        )
        previous_carbon = carbon

    total_tco2e = rows[-1].cumulative_avoided_tco2e
    projected_loss_ha = rows[-1].cumulative_loss_ha
    annual_mean_tco2e = total_tco2e / duration_years

    # Diagnostic. What the same projected loss would be worth if it were spread evenly over the
    # Protect area instead of placed on the highest risk pixels. Ranked allocation normally
    # gives the smaller number, because frontier forest carries less carbon than interior.
    uniform_tco2e = standing_tco2e * safe_pct(projected_loss_ha, protect_ha) / 100.0

    if rate_pct <= 0:
        narrative = (
            "No forest loss was recorded in this project area between 2014 and 2024, so the "
            "baseline projects no further loss and no avoided emissions can be claimed from "
            "protecting the standing forest."
        )
    else:
        narrative = (
            f"Protecting this {ecosystem_label} ecosystem can avoid an estimated "
            f"{total_tco2e:,.0f} tonnes of CO2eq emissions over the project's "
            f"{duration_years} year duration."
        )

    return ComponentResult(
        component=component,
        applicable=True,
        narrative=narrative,
        tables={"annual_projection": rows},
        values={
            "chart_series": "annual_projection",
            "chart_unit": "tCO2e",
            "chart_axis_label": "Cumulative avoided emissions (tCO2e)",
            "total_tco2e": total_tco2e,              # headline big number
            "annual_mean_tco2e": annual_mean_tco2e,
            "duration_years": duration_years,        # recorded so a saved result is reproducible
            "protect_ha": protect_ha,                # measured on the risk grid, not the 4.1 grid
            "protect_risk_coverage_pct": risk_coverage_pct,
            "projected_loss_ha": projected_loss_ha,
            "standing_tco2e": standing_tco2e,        # all carbon on the Protect area
            "baseline_rate_pct": rate_pct,
            "baseline_rate_source": "AOI forest 2014 to 2024, component 1.5",
            "allocation": "descending deforestation risk",
            "uniform_allocation_tco2e": uniform_tco2e,   # diagnostic, see the markdown cell
            "peat_ha": peat_ha,
            "biomass_coverage_pct": biomass_coverage_pct,
            "pools_included": list(BIOMASS_POOLS),
            "ecosystem_ha": ecosystem_ha,            # reference ecosystem split of the Protect pool
            "ecosystem_label": ecosystem_label,      # the word used in the narrative
        },
        flags=flags,
    )

---
## Run and save

In [ ]:
# 5.1 needs the deforestation rate from 1.5. Unlike the optional peat lookup in 3.2, this stage
# is required: without a rate there is no baseline to project.
general = load_results(aoi_id, STAGE_GENERAL)
rate_pct = component_values(general, "1.5").get("rate_pct")

results: dict[str, ComponentResult] = {}

results["5.1"] = analyze_avoided_deforestation_emissions(
    aoi, PROJECT_DURATION_YEARS, rate_pct
)

for key, r in results.items():
    print(f"[{key}] {r.component}{'' if r.applicable else '  (not applicable)'}")
    print(f"      {r.narrative}")
    for f in r.flags:
        print(f"      FLAG: {f}")

In [ ]:
path = save_results(results, aoi, aoi_id, STAGE_BENEFIT)
print(f"Saved {path}")